In [1]:
# https://www.kaggle.com/competitions/essay-gap-aicc-round-2/overview

import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import HTML, display
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from transformers import BertForNextSentencePrediction, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
train = pd.read_csv("/kaggle/input/competitions/essay-gap-aicc-round-2/essay-gap/train.csv")
test = pd.read_csv("/kaggle/input/competitions/essay-gap-aicc-round-2/essay-gap/test.csv")

train.shape, test.shape

((320, 8), (80, 7))

In [3]:
train.head()

,sampleID,before,after,opt_0,opt_1,opt_2,opt_3,label
0,0,The life cycle of a Christmas tree from the se...,One issue that farmers face is the destruction...,The remaining development of the tree greatly ...,The belief in the divinity of Jesus leads to t...,Essentially the recipe brings together what tr...,It is a matter of some debate as to which was ...,0
1,1,Slopes flatter than 25 degrees or steeper than...,The rule of thumb is: A slope that is flat eno...,In her 1850 book The First Christmas in New En...,"In Latin America and the Iberian Peninsula, th...","On steeper slopes, this can occur with as litt...",When the incidence of human triggered avalanch...,3
2,2,"Most workplaces conduct a ""Christmas Party"" so...","Likewise, schools, TAFE (vocational training),...",As many people take their holidays between Chr...,The frequency with which avalanches form in a ...,"In doing so, they employ on-the-ground physica...",The area in and around the basilica begins to ...,0
3,3,The Chronography of 354 illuminated manuscript...,By around 385 the feast for the birth of Jesus...,The eastern inland region where the country is...,In a sermon delivered in Antioch on December 2...,This remains one of the most extensive such ma...,"A cold front, the leading edge of a cooler mas...",1
4,4,English personifications of Christmas were fir...,His character was maintained during the late 1...,"In a sermon in 386, Gregory of Nyssa specifica...",The first evidence of decorated trees associat...,"Following the Restoration in 1660, Father Chri...","In 614, the Persian Sassanid Empire, supported...",2


In [4]:
train['label'].value_counts()

label
2    94
1    94
3    68
0    64
Name: count, dtype: int64

In [5]:
def print_example(row):

    label = f"<span style='color: yellow; font-weight: bold;'>Label: {row['label']} </span>"
    
    options = [
        f"<span style='color: #00bfff;'>{row['before']} </span>"
        f"<span style='color: limegreen;'>{row[f'opt_{i}']}</span>"
        f"<span style='color: #00bfff;'> {row['after']}</span>"
        for i in range(4)
    ]
    
    separator = "<hr style='border: 0.5px solid #ccc; margin: 10px 0;'>"
    
    display(HTML(separator.join([label] + options)))

print_example(train.iloc[6])

In [6]:
def process(df):
    for col in [f'opt_{i}' for i in range(4)] + ['before', 'after']:
        df[f'{col}_last_sent'] = df[col].map(lambda x: sent_tokenize(x)[-1])
        df[f'{col}_first_sent'] = df[col].map(lambda x: sent_tokenize(x)[0])
    
process(train)
process(test)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [8]:
model_name = 'google-bert/bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForNextSentencePrediction.from_pretrained(model_name).to(device).eval()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForNextSentencePrediction LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
@torch.inference_mode()
def get_pred_features(row, device=device):
    sentences_A = [row['before_last_sent']] * 4 + [row[f'opt_{i}_first_sent'] for i in range(4)]
    sentences_B = [row[f'opt_{i}_last_sent'] for i in range(4)] + [row['after_first_sent']] * 4
    
    inputs = tokenizer(
        sentences_A, 
        sentences_B, 
        padding=True, 
        truncation=False, 
        return_tensors='pt'
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    probs = torch.softmax(model(**inputs).logits, dim=-1)[:, 0].cpu().numpy()
    return probs

In [10]:
train, valid = train_test_split(train, stratify=train['label'], test_size=0.1, random_state=42)
train, valid = train.reset_index(drop=True), valid.reset_index(drop=True)
train.shape, valid.shape

((288, 20), (32, 20))

In [11]:
X_train = np.stack([get_pred_features(train.iloc[i]) for i in tqdm(range(len(train)))])
X_valid = np.stack([get_pred_features(valid.iloc[i]) for i in tqdm(range(len(valid)))])
X_test = np.stack([get_pred_features(test.iloc[i]) for i in tqdm(range(len(test)))])

y_train, y_valid = train['label'].values, valid['label'].values

X_train.shape, X_valid.shape, X_test.shape

  0%|          | 0/288 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

((288, 8), (32, 8), (80, 8))

In [12]:
score_train = f1_score(y_train, (X_train[:, :4] + X_train[:, 4:]).argmax(axis=-1), average='macro')
score_valid = f1_score(y_valid, (X_valid[:, :4] + X_valid[:, 4:]).argmax(axis=-1), average='macro')

score_train, score_valid

(0.9438492105430027, 1.0)

In [13]:
preds = (X_test[:, :4] + X_test[:, 4:]).argmax(axis=-1)

submission = pd.DataFrame({"sampleID": test["sampleID"], "answer": preds})
submission.head()

,sampleID,answer
0,100,0
1,101,0
2,102,2
3,103,0
4,104,3


In [14]:
submission.to_csv("submission.csv", index=False)

In [15]:
submission['answer'].value_counts()

answer
3    25
0    24
2    19
1    12
Name: count, dtype: int64